# 🧬 Project 5 — Semantic Correspondence (COMPLETE PIPELINE)
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q

# Scaricamento Dataset
!python dataloaders/download_spair.py --root ./data
!python dataloaders/download_pfpascal.py --root ./data

import os, torch, gc
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print('[INFO] GPU Cleared.')

## 🔍 1. Baseline Evaluation (Multi-Backbone)

In [ ]:
# 1.1 DINOv2
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt

In [ ]:
# 1.2 DINOv3 (Reg)
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov3 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov3.txt

In [ ]:
# 1.3 SAM (ViT-B)
clear_gpu()
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone sam_vitb --batch_size 1 --results_file /content/drive/MyDrive/AML/Results/baseline_sam.txt

## 🚀 2. Training Stage (PEFT Comparison)

In [ ]:
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
# 2.1 Training LoRA
if not os.path.exists(f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir "$DRIVE_CKPTS/lora_only"
else: print('[OK] LoRA trovato.')

In [ ]:
# 2.2 Training BitFit
if not os.path.exists(f'{DRIVE_CKPTS}/bitfit_only/bitfit_only_best.pth'):
    !python train.py --peft_type bitfit --dataset_root ./data/SPair-71k --epochs 5 --exp_name bitfit_only --output_dir ./checkpoints/bitfit_only --backup_dir "$DRIVE_CKPTS/bitfit_only"
else: print('[OK] BitFit trovato.')

## 🎯 3. Ablation Study: Adaptive Window (AW)

In [ ]:
# 3.1 LoRA: No AW vs With AW
print('--- LoRA Only ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/lora_no_aw.txt
print('--- LoRA + AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/lora_aw.txt

In [ ]:
# 3.2 BitFit: No AW vs With AW
print('--- BitFit Only ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --no_adaptive_win --results_file /content/drive/MyDrive/AML/Results/bitfit_no_aw.txt
print('--- BitFit + AW ---')
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/bitfit_aw.txt

## 🌍 4. Generalization: PF-Pascal

In [ ]:
PASCAL_ROOT = './data/PF-Pascal/PF-Pascal'
print('--- PF-Pascal: LoRA ---')
!python evaluate.py --dataset_root $PASCAL_ROOT --dataset_type pfpascal --checkpoint "$DRIVE_CKPTS/lora_only/lora_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/gen_pascal_lora.txt
print('--- PF-Pascal: BitFit ---')
!python evaluate.py --dataset_root $PASCAL_ROOT --dataset_type pfpascal --checkpoint "$DRIVE_CKPTS/bitfit_only/bitfit_only_best.pth" --results_file /content/drive/MyDrive/AML/Results/gen_pascal_bitfit.txt

## ⚖️ 5. Visual Demos

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

In [ ]:
launch_robustness_demo(ckpt_name='lora_only')